# Do Games Actually Move the Market?
### Critical Reception, Sales, and Publisher Stock Performance Around Launch Windows

---
## Contents
1. Introduction
2. Hypothesis
3. Data Sources & Methodology
4. Environment Setup
5. Data collection
    - i. Games
    - ii. Stock Dataset
6. Data Cleaning
7. Exploratory Analysis
    - i. Score Distribution
    - ii. Score vs Stock Return
    - iii. Returns by Score Tier
8. Findings
9. Limitations

---
## 1. Introduction

The games industry generated over $180 bllion in revenue in 2023 - larger than film and music combined. Yet despite its scale, the relationship between individual game performance and publisher stock price remains poorly understood outside of specialist analysts.

This notebook investigates whether measurable game performance indicators - critic review scores and sales figures - correlate with short-term abnormal movement in publisher stock prices.

The publishers under analysis are **Electronic Arts (EA)**, **Activision Blizzard (ATVI)**, and **Ubisoft (UBI.PA)** - three of the largest Western publishers with consistent release histories and publicly traded stock across the period of interest.

---
## 2. Hypothesis

I propose two related hypotheses:

**H1: Review scores:** Games with significantly above-average Metacritic scores (≥ 85) will correlate 
with positive abnormal stock returns in the ±7 day window around launch. Critically reviewed flops 
(≤ 50) will correlate with negative movement.

**H2: Sales performance:** Titles that outperform sales expectations relative to franchise history 
will produce stronger positive returns than review scores alone.

**Expected complication:** For heavily anticipated titles with large marketing spend - think a mainline 
FIFA, Call of Duty, or Assassin's Creed - I expect the market to have already priced in expected 
performance, dampening the post-launch signal. This is the *efficient market hypothesis* applied to 
game launches, and testing whether it holds is part of what makes this interesting.

---
## 3. Data Sources and Methodology

**Game data:** Metacritic scores and release dates sourced from the RAWG Video Games Database API 
(free tier, no scraping required). Filtered to EA, Activision Blizzard, and Ubisoft titles released 
between 2012–2023.

**Sales data:** Where available, pulled from public VGChartz records via a cleaned community dataset. 
Sales figures are estimates and treated as directional rather than precise.

**Stock data:** Daily adjusted closing prices fetched via `yfinance` for EA (`EA`), 
Activision Blizzard (`ATVI`), and Ubisoft (`UBI.PA`) across the same period.

**Event window:** I define launch impact as the stock return across a **±7 day window** centred on 
release date. This is a simplified event study - in academic finance, abnormal returns are typically 
benchmarked against a market index, but for exploratory analysis we use raw return as a readable proxy.

**Limitations acknowledged upfront:** Sales data has gaps. Metacritic scores reflect critic consensus, 
not audience reception (which can diverge sharply). Market movements are noisy and multi-causal - 
a good game launch can coincide with a bad earnings report. This analysis is exploratory, not causal.

---
## 4. Environment Setup

Libraries used:
* `requests` for API calls
* `pandas` and `numpy` for data manipulation
* `matplotlib` and `seaborn` for visualisation
* `yfinance` for stock data
* `python-dotenv`to load the API key from a local `.env` file.

In [3]:
import os
import time
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv

load_dotenv()
RAWG_API_KEY = os.getenv("RAWG_API_KEY")

---
## 5. Data Collection

First, I pulled game data from the RAWG API, querying by publisher slug and filtering for titles 
with Metacritic scores. Each publisher is fetched across multiple pages to build a representative 
sample. Then, I pulled adjusted daily closing prices for each corresponding stock ticker via yfinance.

The two datasets are not directly joinable by a shared key - they are linked by publisher identity 
and date proximity. The join happens at the event window calculation stage.

In [6]:
def fetch_games(publisher, api_key, pages=5):
    games = []

    for page in range(1, pages + 1):
        url = "https://api.rawg.io/api/games"
        params = {
            "key": api_key,
            "publishers": publisher,
            "page": page,
            "page_size": 40,
            "metacritic": "1,100",
            "ordering": "-metacritic"
        }

        response = requests.get(url, params=params)
        data = response.json()
        for game in data.get("results", []):
            games.append({
                "title": game["name"],
                "release_date": game["released"],
                "metacritic": game["metacritic"],
                "publisher": publisher
            })
        time.sleep(0.3)
    return games

publishers = {
    "electronic-arts": "EA",
    "activision": "ATVI",
    "ubisoft": "UBI.PA"
}

all_games = []
for slug, ticker in publishers.items():
    games = fetch_games(slug, RAWG_API_KEY)

    for g in games:
        g["ticker"] = ticker
    all_games.extend(games)

df_games = pd.DataFrame(all_games)
df_games["release_date"] = pd.to_datetime(df_games["release_date"])
df_games = df_games.dropna(subset=["metacritic", "release_date"])
df_games = df_games[(df_games["release_date"] >= "2013-01-01") & 
                    (df_games["release_date"] <= "2025-12-31")]

print(df_games.shape)
df_games.head()


(48, 5)


,title,release_date,metacritic,publisher,ticker
29,It Takes Two,2021-03-26,88,electronic-arts,EA
34,Battlefield 1,2016-10-21,88,electronic-arts,EA
35,Mass Effect: Legendary Edition,2021-05-14,88,electronic-arts,EA
41,Titanfall 2,2016-10-28,87,electronic-arts,EA
48,Dragon Age: Inquisition,2014-11-18,86,electronic-arts,EA
